In [ ]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [ ]:
save_folder = 'run7'
n_points = 5000

lower_factor = 0.99
upper_factor = 2 - lower_factor

In [ ]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:

b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [ ]:
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [ ]:

def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):

    def integrand(y, x, mg, a1, a2, m2_func, q_val):
        k        = sqrt_s * x
        phi      = 2*np.pi*y
        jacobian = 2*np.pi*sqrt_s
        return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
                  T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian
    def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q_val),
                0, 1, n=n_points
            )[0]

    integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
    
    return integral_value

In [ ]:
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [ ]:
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')
chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)
chi2_total = chi2_7 + chi2_8 + chi2_13
minuit_born = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_born.migrad()
minuit_born.hesse()


In [ ]:
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001
    n_points = 5000

    # def integrand(y, x, mg, a1, a2, m2_func, q_val):
    #     k        = sqrt_s * x
    #     phi      = 2*np.pi*y
    #     jacobian = 2*np.pi*sqrt_s
    #     return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
    #               T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2

        # def inner_integral(x):
        #     return fixed_quad(
        #         lambda y: integrand(y, x, mg, a1, a2, mg_model, q2),
        #         0, 1, n=n_points
        #     )[0]

        # integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]

        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)


        diff_T = integral_value
        # print(f"q2: {q2}, diff_T: {diff_T}")

        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}


In [ ]:

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    minuit_born.values['eps'],
    minuit_born.values['mg'], 
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


In [ ]:
def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))



In [ ]:
fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#-----------------------------------------------------------------------------------------------

#-----------------------------------------------------------------------------------------------

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [ ]:
#=============================================================
# PLOT BORN SIGMA TOT 
#=============================================================


# data_sigma_tot_atlas = pd.read_csv(
#     "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
#     delim_whitespace=True,
#     header=None,
#     nrows=70
# )

# data_sigma_tot_totem = pd.read_csv(
#     "../../../data/sigma_tot_2/ensemble_StRh_totem.dat",  # Supondo que existe um arquivo similar para TOTEM
#     delim_whitespace=True,
#     header=None,
#     nrows=80
# )

# x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
# y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
# y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

# x_sigma_tot_totem = data_sigma_tot_totem[0].to_numpy()
# y_sigma_tot_totem = data_sigma_tot_totem[1].to_numpy()
# y_error_sigma_tot_totem = data_sigma_tot_totem[2].to_numpy()

# start_sqrt_s = 1
# max_sqrt_s = 13010
# step = 100

# def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
#     fig.add_trace(go.Scatter(
#         x=x,
#         y=y,
#         mode='lines+markers',
#         line=dict(color=color, width=width, dash=line_style),
#         marker=dict(size=size),
#         name = label, 
#         showlegend=legend
#     ))

# def get_sigma_tot(epsilon, mg, a1, a2, mg_model):

#     lst_sigma_tot = []
#     lst_sqrt_s = []

#     sqrt_s = start_sqrt_s

#     while sqrt_s <= max_sqrt_s:
#         def integrand(y, x, mg, a1, a2, m2_func):
#             k = sqrt_s * x
#             phi = 2 * np.pi * y
#             jacobian = 2 * np.pi * sqrt_s

#             return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian
        
#         def amp_calculation(diff_T, s, epsilon):
#             alpha_pomeron = 1.0 + epsilon
#             regge_factor = (s / s0) ** alpha_pomeron
            
#             return 1j * 8.0 * regge_factor * diff_T

#         s = sqrt_s ** 2

#         integral_value = fixed_quad(
#             lambda x: fixed_quad(
#                 lambda y: integrand(y, x, mg, a1, a2, mg_model),
#                 0, 1, n=n_points)[0],

#             0, 1, n=n_points)[0]
        
#         lst_sigma_tot.append(sigma_tot(
#             amp_calculation(integral_value, s, epsilon), s))
        
#         lst_sqrt_s.append(sqrt_s)
#         sqrt_s += step
#     return lst_sigma_tot, lst_sqrt_s

# -----------------------------------------------------------------------------------------------

# sigma_tot_pl_atlas = get_sigma_tot(
#     m_pl_atlas.values['eps'],
#     m_pl_atlas.values['mg'],
#     m_pl_atlas.values['a1'],
#     m_pl_atlas.values['a2'],
#     m2_pl
# )

# sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
# lst_sqrt_s = sigma_tot_pl_atlas[1]



# fig = go.Figure()

# add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

# #-----------------------------------------------------------------------------------------------
# #----

# add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')
# add_data_trace(fig, x_sigma_tot_totem, y_sigma_tot_totem, y_error_sigma_tot_totem, name='TOTEM', show_label=True, mode='markers')

# fig.update_layout(
#     title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
#     xaxis=dict(
#         title='√s [GeV]',
#         type='log',
#         range=[np.log10(2000), np.log10(14000)],
#     ),
#     yaxis=dict(
#         title='σ_tot [mb]',
#         range=[80, 125]
#     ),
#     showlegend=True,
#     legend=dict(
#         title='Ensembles'
#     ),
#     plot_bgcolor='white',
#     hovermode='x unified'
# )
    
# fig.update_xaxes(gridcolor='lightgray')
# fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")

In [ ]:
lst_q_integration = np.linspace(0, 0.2, 100)
lst_b_integration = np.linspace(0, 30, 100)

In [ ]:
import numpy as np
from scipy.integrate import fixed_quad
from scipy.special import j0


# Wrapper functions to separate real and imaginary parts for chi integral
def chi_integrand_real(q, b_value, sqrt_s, mg, a1, a2, m2_pl, eps):
    s = sqrt_s ** 2
    integral_value = full_int(mg, a1, a2, m2_pl, q, sqrt_s)
    t = -(q**2)
    born_amp = amp_calculation(integral_value, s, eps, t)
    chi_value = (q * j0(b_value * q) * born_amp) / s
    return np.real(chi_value)

def chi_integrand_imag(q, b_value, sqrt_s, mg, a1, a2, m2_pl, eps):
    s = sqrt_s ** 2
    integral_value = full_int(mg, a1, a2, m2_pl, q, sqrt_s)
    t = -(q**2)
    born_amp = amp_calculation(integral_value, s, eps, t)
    chi_value = (q * j0(b_value * q) * born_amp) / s
    return np.imag(chi_value)

# Wrapper functions for eikonal integral
def eik_integrand_real(b, q_exp, sqrt_s, chi_sum_complex):
    s = sqrt_s ** 2
    factor = 1 - np.exp(1j * chi_sum_complex)
    eik_value = b * j0(b * q_exp) * factor * (1j * s)
    return np.real(eik_value)

def eik_integrand_imag(b, q_exp, sqrt_s, chi_sum_complex):
    s = sqrt_s ** 2
    factor = 1 - np.exp(1j * chi_sum_complex)
    eik_value = b * j0(b * q_exp) * factor * (1j * s)
    return np.imag(eik_value)


def model_function_eik(x_born, eps, mg, a1, a2, sqrt_s, model_type='log',
                   q_min=0.0, q_max=0.2, b_min=0.0, b_max=30.0, n_points=5000):
    """
    Calculate differential cross-section using fixed_quad integration.
    
    Parameters:
    -----------
    x_born : array-like
        Experimental points (momentum transfer values)
    eps, mg, a1, a2 : float
        Model parameters
    sqrt_s : float
        Center of mass energy (fixed for all x_born points)
    model_type : str
        Model type identifier (default: 'log')
    q_min, q_max : float
        Integration limits for q (momentum transfer)
    b_min, b_max : float
        Integration limits for b (impact parameter)
    n_points : int
        Number of points for fixed_quad integration
    
    Returns:
    --------
    numpy.array
        Differential cross-sections for each point in x_born
    """
    
    s = sqrt_s ** 2
    results = []
    
    # Loop over experimental points (for each q_exp in x_born)
    for q_exp in x_born:
        
        # Define b integrand functions for this specific q_exp
        def b_integrand_real(b):
            # For each b, integrate over q to get chi
            chi_real, _ = fixed_quad(
                lambda q: chi_integrand_real(q, b, sqrt_s, mg, a1, a2, m2_pl, eps),
                q_min, q_max, n=n_points
            )
            
            chi_imag, _ = fixed_quad(
                lambda q: chi_integrand_imag(q, b, sqrt_s, mg, a1, a2, m2_pl, eps),
                q_min, q_max, n=n_points
            )
            
            chi_sum = chi_real + 1j * chi_imag
            return eik_integrand_real(b, q_exp, sqrt_s, chi_sum)
        
        def b_integrand_imag(b):
            # For each b, integrate over q to get chi
            chi_real, _ = fixed_quad(
                lambda q: chi_integrand_real(q, b, sqrt_s, mg, a1, a2, m2_pl, eps),
                q_min, q_max, n=n_points
            )
            
            chi_imag, _ = fixed_quad(
                lambda q: chi_integrand_imag(q, b, sqrt_s, mg, a1, a2, m2_pl, eps),
                q_min, q_max, n=n_points
            )
            
            chi_sum = chi_real + 1j * chi_imag
            return eik_integrand_imag(b, q_exp, sqrt_s, chi_sum)
        
        # Final integration over b for this q_exp
        eik_real, _ = fixed_quad(b_integrand_real, b_min, b_max, n=n_points)
        eik_imag, _ = fixed_quad(b_integrand_imag, b_min, b_max, n=n_points)
        
        eik_amp_sum = eik_real + 1j * eik_imag
        
        # Calculate differential cross-section
        diff_sigma = differential_sigma(eik_amp_sum, s)
        results.append(diff_sigma)
    
    return np.array(results)

In [ ]:
def model_7(x, eps, mg, a1, a2):
    return model_function_eik(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)

chi2_total = chi2_7 

minuit_eik = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_eik.limits["mg"] = (0, 0.9)
minuit_eik.limits["eps"] = (0, 1)
minuit_eik.limits["a1"] = (0, 2)
minuit_eik.limits["a2"] = (0, 2)


minuit_eik.migrad()
minuit_eik.migrad()
minuit_eik.hesse()
